In [ ]:
# Colab Setup - Run this cell first if using Google Colab
# Skip this cell if running locally with torchref already installed

#install torchref
!pip install torchref

# Download a structure/dataset pair 
!wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.pdb
!wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.mtz

pdb_file = "./1DAW.pdb"
mtz_file = "./1DAW.mtz"

# TorchRef Basic Usage

This notebook demonstrates the basic usage of TorchRef for crystallographic refinement.

## Setup

In [9]:
import torchref

## Loading Structures

TorchRef provides two model classes:
- `Model`: Basic model for atomic coordinates
- `ModelFT`: Model with structure factor calculation capability

In [10]:
from torchref.model import Model, ModelFT

pdb_file = "./1DAW.pdb"
mtz_file = "./1DAW.mtz"

# Basic model
model = Model().load_pdb(pdb_file)

# Model with structure factor calculation
model_ft = ModelFT().load_pdb(pdb_file)

Loaded 3051 atoms
Loaded 3051 atoms


## Loading Reflection Data

In [11]:
from torchref.io import ReflectionData

reflection_data = ReflectionData().load_mtz(mtz_file)

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged


## Scaling

The `Scaler` class handles bulk solvent correction and scale factor optimization.

In [12]:
from torchref.scaling import Scaler

scaler = Scaler(model=model_ft, data=reflection_data)
scaler.initialize()

print(f"Before refinement: Rwork={scaler.rfactor()[0]:.4f}, Rfree={scaler.rfactor()[1]:.4f}")

scaler.refine_lbfgs()                        

print(f"After refinement:  Rwork={scaler.rfactor()[0]:.4f}, Rfree={scaler.rfactor()[1]:.4f}")

Initialized ScalerBase with 20 bins.
Parametrization built for 6 unique atom types
Calculating initial scale factors using 20 bins.


/das/work/p17/p17490/Peter/testing_packages/test_torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


Before refinement: Rwork=0.2891, Rfree=0.3357
Refining scales with LBFGS...
Scale refinement complete. rwork: 0.2098, rfree: 0.2738

Final Scale Parameters: 
  log_scale: tensor([-6.6597, -6.5830, -6.5546, -6.4989, -6.4868, -6.4581, -6.4191, -6.3839,
        -6.3798, -6.3414, -6.3183, -6.2742, -6.2202, -6.1759, -6.1353, -6.0684,
        -5.9910, -5.9808, -5.9974, -6.0212])
  U: tensor([-0.2640, -0.1745, -0.0897, -0.0036, -0.1583, -0.0029])
  solvent.log_k_solvent: -0.9674586653709412
  solvent.b_solvent: 46.07955551147461
  solvent.phase_offset: -0.001046499121002853


/das/work/p17/p17490/Peter/testing_packages/test_torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


After refinement:  Rwork=0.2098, Rfree=0.2738


## Setting Up Refinement, low level functions

The `LBFGSRefinement` class provides a complete refinement workflow.
This is demonstrated in the code examples. 

Here we show how the refinement is done at a lower level.

In [13]:
from torchref.refinement import LBFGSRefinement

refinement = LBFGSRefinement(pdb=pdb_file, data_file=mtz_file)

print(f"Initial R-factors: Rwork={refinement.get_rfactor()[0]:.4f}, Rfree={refinement.get_rfactor()[1]:.4f}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 10 bins.
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disulfide bonds: 

/das/work/p17/p17490/Peter/testing_packages/test_torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


## Perturbing the Model

Shake coordinates to simulate a starting model with errors.

In [14]:
refinement.model.shake_coords(0.1)  # Shake by 0.1 Angstroms

print(f"After shaking: Rwork={refinement.get_rfactor()[0]:.4f}, Rfree={refinement.get_rfactor()[1]:.4f}")

After shaking: Rwork=0.2330, Rfree=0.2819


## Loss State and Weights

The `LossState` object manages targets and their weights for refinement.

In [15]:
loss_state = refinement.create_loss_state()
refinement.add_target_info_to_state(loss_state)
refinement.populate_state_meta(loss_state)
refinement.update_weights(loss_state)

print("Weights:")
for name, weight in loss_state.weights.items():
    print(f"  {name}: {weight:.4f}")

print(f"\nTotal loss: {loss_state.aggregate().item():.2f}")

Weights:
  xray: 0.5210
  geometry/bond: 1.1182
  geometry/angle: 1.2594
  geometry/torsion: 1.0632
  geometry/planarity: 10.0000
  geometry/chiral: 10.0000
  geometry/nonbonded: 1.0495
  adp/simu: 2.4221
  adp/locality: 1.2742
  adp/KL: 1.3165
  geometry: 10.0000
  adp: 10.0000

Total loss: 482.39


## Running Refinement (CPU)

In [16]:
parameters = refinement.parameters()
refinement._optimize_lbfgs(loss_state, parameters, max_iter=100, nsteps=1)

print(f"After refinement: Rwork={refinement.get_rfactor()[0]:.4f}, Rfree={refinement.get_rfactor()[1]:.4f}")

After refinement: Rwork=0.1720, Rfree=0.2978


In [17]:
print(refinement.collect_metrics())

{'rwork': 0.17202423512935638, 'rfree': 0.2978192865848541, 'rfree_gap': 0.12579505145549774, 'geometry': {'bond': {'loss': -3.4662365913391113, 'n': 2837, 'rms_delta': 0.0018579016905277967, 'rms_z': 0.1269371062517166, 'mean_sigma': 0.012758337892591953}, 'angle': {'loss': -2.4080870151519775, 'n': 3828, 'rms_delta': 1.1688019037246704, 'rms_z': 0.5036155581474304, 'mean_sigma': 1.8857680559158325}, 'torsion': {'loss': -0.5113788843154907, 'n': 1074, 'rms_delta': 8.201577186584473, 'rms_z': 0.8203648328781128, 'mean_sigma': 9.892923355102539}, 'planarity': {'loss': -2.326472520828247, 'n': 3149, 'rms_delta': 0.0014100793050602078, 'rms_z': 0.016974225640296936, 'mean_sigma': 0.05312797427177429}, 'chiral': {'loss': -0.6902055740356445, 'n': 407, 'rms_delta': 0.004848336800932884, 'rms_z': 0.02424168586730957, 'mean_sigma': 0.20000001788139343}, 'nonbonded': {'loss': 0.0571637824177742, 'n': 28724, 'n_violations': 2644, 'rms_violation': 0.40670761466026306, 'max_violation': 3.41596436

## GPU Acceleration

Move the refinement to GPU for faster computation.

In [18]:
import torch

if torch.cuda.is_available():
    refinement.cuda()
    loss_state.cuda()
    
    # Run refinement on GPU
    parameters = refinement.parameters()
    refinement._optimize_lbfgs(loss_state, parameters, max_iter=100, nsteps=1)
    
    print(f"After GPU refinement: Rwork={refinement.get_rfactor()[0]:.4f}, Rfree={refinement.get_rfactor()[1]:.4f}")
else:
    print("CUDA not available")

Model moved to device: cuda
After GPU refinement: Rwork=0.1553, Rfree=0.3119
